In [1]:
import numpy as np
import pandas as pd

In [16]:
fraud_df=pd.read_csv('../data/raw/Fraud_Data.csv')

In [17]:
fraud_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [18]:
# convert timestamp
fraud_df["purchase_time"] = pd.to_datetime(
    fraud_df["purchase_time"]
)

fraud_df = fraud_df.sort_values(
    ["user_id", "purchase_time"]
)

In [ ]:
# Time between signup and purchase
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

fraud_df['time_to_purchase_sec'] = (
    fraud_df['purchase_time'] - fraud_df['signup_time']
).dt.total_seconds()

In [20]:
fraud_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,time_to_purchase_sec
116708,2,2015-01-11 03:47:13,2015-02-21 10:03:37,54,FGBQNDNBETFJJ,SEO,Chrome,F,25,8.802175e+08,0,3564984.0
15108,4,2015-06-02 16:40:57,2015-09-26 21:32:16,41,MKFUIVOHLJBYN,Direct,Safari,F,38,2.785906e+09,0,10039879.0
46047,8,2015-05-28 07:53:06,2015-08-13 11:53:07,47,SCQGQALXBUQZJ,SEO,Chrome,M,25,3.560567e+08,0,6667201.0
67650,9,2015-05-16 15:58:32,2015-05-20 23:06:42,62,IEZOHXPZBIRTE,SEO,FireFox,M,21,7.591047e+08,0,371290.0
109067,12,2015-01-10 06:25:12,2015-03-04 20:56:37,35,MSNWCFEHKTIOY,Ads,Safari,M,19,2.985180e+09,0,4631485.0


In [ ]:
# Transactions per device
device_tx_count = fraud_df.groupby('device_id').size()

fraud_df['transactions_per_device'] = (
    fraud_df['device_id'].map(device_tx_count)
)

device_id
AAALBGNHHVMKG     1
AAAWIHVCQELTP     1
AAAXJHWCLISKY     1
AAAXXOZJRZRAO    11
AABFGRPBQHWFQ     1
dtype: int64

In [25]:
# Accounts per IP address
ip_count = fraud_df.groupby('ip_address').size()

fraud_df['accounts_per_ip'] = (
    fraud_df['ip_address'].map(ip_count)
)
ip_count.head()

ip_address
52093.496895     1
93447.138961     1
105818.501505    1
117566.664867    1
131423.789042    1
dtype: int64

In [26]:
# Time-window velocity by device or IP
fraud_df = fraud_df.sort_values(
    ['device_id', 'purchase_time']
)

fraud_df['seconds_since_prev_device_tx'] = (
    fraud_df.groupby('device_id')['purchase_time']
    .diff()
    .dt.total_seconds()
)

In [27]:
print("Unique users:", fraud_df['user_id'].nunique())
print("Rows:", len(fraud_df))

print("Repeated devices:",
      (fraud_df['device_id'].value_counts() > 1).sum())

print("Repeated IPs:",
      (fraud_df['ip_address'].value_counts() > 1).sum())

Unique users: 151112
Rows: 151112
Repeated devices: 6175
Repeated IPs: 760


User-level transaction frequency and velocity features cannot be computed because each user has exactly one transaction.

In [29]:
# Time-to-purchase feature (highly recommended)
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])

fraud_df['time_to_purchase_hours'] = (
    fraud_df['purchase_time'] - fraud_df['signup_time']
).dt.total_seconds() / 3600

fraud_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,time_to_purchase_sec,transactions_per_device,accounts_per_ip,seconds_since_prev_device_tx,time_to_purchase_hours
34321,226648,2015-05-12 16:00:31,2015-06-13 23:42:18,45,AAALBGNHHVMKG,Direct,Chrome,M,34,2.494581e+09,0,2792507.0,1,1,NaN,775.696389
57616,103319,2015-02-05 22:20:04,2015-03-29 00:39:07,94,AAAWIHVCQELTP,Ads,Chrome,M,29,8.092637e+08,0,4414743.0,1,1,NaN,1226.317500
46520,36633,2015-05-10 00:49:53,2015-07-22 03:18:01,46,AAAXJHWCLISKY,Ads,IE,M,40,2.891497e+06,0,6316088.0,1,1,NaN,1754.468889
69175,325729,2015-01-01 04:25:22,2015-01-01 04:25:23,57,AAAXXOZJRZRAO,Ads,FireFox,F,36,1.377849e+09,1,1.0,11,11,NaN,0.000278
35860,64674,2015-01-01 04:25:23,2015-01-01 04:25:24,57,AAAXXOZJRZRAO,Ads,FireFox,F,36,1.377849e+09,1,1.0,11,11,1.0,0.000278


In [30]:
# Hour of day
fraud_df['hour_of_day'] = (
    fraud_df['purchase_time'].dt.hour
)
# Day of the week
fraud_df['day_of_week'] = (
    fraud_df['purchase_time'].dt.dayofweek
)
fraud_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,time_to_purchase_sec,transactions_per_device,accounts_per_ip,seconds_since_prev_device_tx,time_to_purchase_hours,hour_of_day,day_of_week
34321,226648,2015-05-12 16:00:31,2015-06-13 23:42:18,45,AAALBGNHHVMKG,Direct,Chrome,M,34,2.494581e+09,0,2792507.0,1,1,NaN,775.696389,23,5
57616,103319,2015-02-05 22:20:04,2015-03-29 00:39:07,94,AAAWIHVCQELTP,Ads,Chrome,M,29,8.092637e+08,0,4414743.0,1,1,NaN,1226.317500,0,6
46520,36633,2015-05-10 00:49:53,2015-07-22 03:18:01,46,AAAXJHWCLISKY,Ads,IE,M,40,2.891497e+06,0,6316088.0,1,1,NaN,1754.468889,3,2
69175,325729,2015-01-01 04:25:22,2015-01-01 04:25:23,57,AAAXXOZJRZRAO,Ads,FireFox,F,36,1.377849e+09,1,1.0,11,11,NaN,0.000278,4,3
35860,64674,2015-01-01 04:25:23,2015-01-01 04:25:24,57,AAAXXOZJRZRAO,Ads,FireFox,F,36,1.377849e+09,1,1.0,11,11,1.0,0.000278,4,3


In [35]:
fraud_df['time_since_signup_sec'] = (
    fraud_df['purchase_time'] - fraud_df['signup_time']
).dt.total_seconds()
fraud_df['time_since_signup_sec']

34321    2792507.0
57616    4414743.0
46520    6316088.0
69175          1.0
35860          1.0
           ...    
3558      545000.0
64382    5989393.0
84364    2466625.0
84217    7599013.0
54293    1295934.0
Name: time_since_signup_sec, Length: 151112, dtype: float64

Data transformation

In [ ]:
categorical_features = [
    'source',
    'browser',
    'sex'
]

fraud_df = pd.get_dummies(
    fraud_df,
    columns=categorical_features,
    drop_first=True
)

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    'source',
    'browser',
    'sex',
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(drop='first', handle_unknown='ignore'),
            categorical_features
        )
    ],
    remainder='passthrough'
)

In [44]:
cols_to_drop = [
    'user_id',
    'device_id',
    'time_to_purchase_hours',
]

fraud_df = fraud_df.drop(columns=cols_to_drop)

In [47]:
X = fraud_df.drop('class', axis=1)
y = fraud_df['class']

In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [49]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = [
    'age',
    'purchase_value',
    'time_since_signup_sec',
    'accounts_per_ip',
    'hour_of_day',
]

X_train[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)

X_test[num_cols] = scaler.transform(
    X_test[num_cols]
)

In [50]:
print(y_train.value_counts())

class
0    109568
1     11321
Name: count, dtype: int64


In [53]:
X_train = X_train.drop(
    columns=[
        'signup_time',
        'purchase_time',
        'time_since_signup'
    ]
)

X_test = X_test.drop(
    columns=[
        'signup_time',
        'purchase_time',
        'time_since_signup'
    ]
)

In [54]:
bool_cols = X_train.select_dtypes(include='bool').columns

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [55]:
print(X_train.dtypes.unique())

[dtype('float64') dtype('int64') dtype('int32')]


In [58]:
X_train['seconds_since_prev_device_tx'] = (
    X_train['seconds_since_prev_device_tx']
    .fillna(-1)
)

X_test['seconds_since_prev_device_tx'] = (
    X_test['seconds_since_prev_device_tx']
    .fillna(-1)
)

In [59]:
X_train.isna().sum().sort_values(ascending=False)

purchase_value                  0
age                             0
ip_address                      0
time_to_purchase_sec            0
transactions_per_device         0
accounts_per_ip                 0
seconds_since_prev_device_tx    0
hour_of_day                     0
day_of_week                     0
time_since_signup_sec           0
source_Direct                   0
source_SEO                      0
browser_FireFox                 0
browser_IE                      0
browser_Opera                   0
browser_Safari                  0
sex_M                           0
dtype: int64

In [60]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(y_train_smote.value_counts())


class
0    109568
1    109568
Name: count, dtype: int64


In [ ]:
# class distribution before and after resampling
comparison = pd.DataFrame({
    "Before_SMOTE": y_train.value_counts(),
    "After_SMOTE": y_train_smote.value_counts()
})

print(comparison)

       Before_SMOTE  After_SMOTE
class                           
0            109568       109568
1             11321       109568
